## Imports

In [1]:
import os
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

## Config

In [2]:
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME  = "bert-base-uncased"
MAX_LEN     = 128
BATCH_SIZE  = 16
EPOCHS      = 2
LR          = 2e-5
OUTPUT_DIR  = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
print(f"Using device: {DEVICE}")

Using device: cuda


## Dataset

In [3]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len
 
    def __len__(self):
        return len(self.texts)
 
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }
 
 
def load_data(tokenizer, split_size=2000):
    """Load SST-2 (GLUE) — positive/negative sentiment."""
    print("Loading SST-2 dataset …")
    ds = load_dataset("glue", "sst2")
 
    def extract(split, n):
        items  = ds[split].select(range(n))
        texts  = [x["sentence"] for x in items]
        labels = [x["label"]    for x in items]
        return texts, labels
 
    train_texts, train_labels = extract("train",      split_size)
    val_texts,   val_labels   = extract("validation", 400)
 
    train_ds = SentimentDataset(train_texts, train_labels, tokenizer, MAX_LEN)
    val_ds   = SentimentDataset(val_texts,   val_labels,   tokenizer, MAX_LEN)
    return train_ds, val_ds
 

## Training

In [4]:
def train(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["label"].to(DEVICE)
 
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)
 
 
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["label"].to(DEVICE)
 
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)
 
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
 
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="weighted")
    return acc, f1
 

## Attention Extraction

In [5]:
def get_attention_weights(model, tokenizer, text):
    """
    Returns:
        tokens      : list of token strings
        attentions  : tuple of (num_layers,) each (1, num_heads, seq, seq)
    """
    model.eval()
    enc = tokenizer(
        text,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids      = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
 
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=True,
        )
 
    # Keep only non-padding tokens
    seq_len = int(attention_mask.sum().item())
    tokens  = tokenizer.convert_ids_to_tokens(input_ids[0][:seq_len].cpu())
 
    # attentions: tuple of 12 tensors, each (1, 12, 128, 128)
    attentions = outputs.attentions
    return tokens, attentions, seq_len
 
 
def attention_rollout(attentions, seq_len):
    """
    Compute attention rollout (Abnar & Zuidema, 2020).
    Propagates attention through all layers for a holistic view.
    Returns a (seq_len,) importance vector w.r.t. [CLS].
    """
    # Average across heads for each layer → (seq, seq)
    rollout = torch.eye(seq_len, device=DEVICE)
    for layer_attn in attentions:
        # layer_attn: (1, heads, seq, seq) → mean over heads → (seq, seq)
        avg = layer_attn[0, :, :seq_len, :seq_len].mean(dim=0)
        # Add residual connection
        avg = avg + torch.eye(seq_len, device=DEVICE)
        avg = avg / avg.sum(dim=-1, keepdim=True)
        rollout = avg @ rollout
 
    # Importance of each token w.r.t. [CLS] (row 0)
    importance = rollout[0]
    importance = (importance - importance.min()) / (importance.max() - importance.min() + 1e-9)
    return importance.cpu().numpy()
 

## Gradient-Based Importance

In [6]:
def get_gradient_importance(model, tokenizer, text):
    """
    Gradient × Input importance score for each token.
    Uses the embedding gradients w.r.t. the predicted class logit.
    """
    model.eval()
    enc = tokenizer(
        text,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids      = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    seq_len        = int(attention_mask.sum().item())
 
    # Hook embeddings to capture gradients
    embeddings_ref = {}
 
    def save_embedding_hook(module, input, output):
        output.retain_grad()
        embeddings_ref["emb"] = output
 
    hook = model.bert.embeddings.register_forward_hook(save_embedding_hook)
 
    outputs    = model(input_ids=input_ids, attention_mask=attention_mask)
    pred_class = outputs.logits.argmax(dim=-1).item()
    score      = outputs.logits[0, pred_class]
 
    model.zero_grad()
    score.backward()
    hook.remove()
 
    emb  = embeddings_ref["emb"]        # (1, seq, hidden)
    grad = emb.grad                      # (1, seq, hidden)
 
    # Gradient × Input, L2 norm over hidden dim → (seq,)
    importance = (grad * emb).norm(dim=-1)[0, :seq_len]
    importance = importance.detach().cpu().numpy()
    importance = (importance - importance.min()) / (importance.max() - importance.min() + 1e-9)
 
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0][:seq_len].cpu())
    return tokens, importance, pred_class
 

## Visualizations 

In [7]:
def plot_token_heatmap(tokens, scores, title, filename, method="attention"):
    """Horizontal bar-style heatmap for token importance."""
    fig, ax = plt.subplots(figsize=(max(8, len(tokens) * 0.55), 2.5))
 
    cmap   = plt.cm.RdYlGn if method == "attention" else plt.cm.YlOrRd
    colors = [cmap(s) for s in scores]
 
    for i, (tok, col) in enumerate(zip(tokens, colors)):
        ax.barh(0, 1, left=i, color=col, edgecolor="white", linewidth=0.5)
        ax.text(i + 0.5, 0, tok, ha="center", va="center",
                fontsize=8, fontweight="bold", color="black")
 
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=mcolors.Normalize(0, 1))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, orientation="horizontal", pad=0.4, fraction=0.04)
 
    ax.set_xlim(0, len(tokens))
    ax.set_ylim(-0.6, 0.6)
    ax.axis("off")
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
 
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved → {path}")
 
 
def plot_attention_head_heatmap(attentions, tokens, layer=11, head=0, filename="attn_head.png"):
    """2-D attention matrix for a specific layer/head."""
    seq_len  = len(tokens)
    attn_mat = attentions[layer][0, head, :seq_len, :seq_len].cpu().numpy()
 
    fig, ax = plt.subplots(figsize=(max(6, seq_len * 0.5), max(5, seq_len * 0.45)))
    sns.heatmap(
        attn_mat, xticklabels=tokens, yticklabels=tokens,
        cmap="Blues", ax=ax, linewidths=0.3, linecolor="lightgrey",
    )
    ax.set_title(f"Attention Weights — Layer {layer+1}, Head {head+1}", fontsize=11)
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", rotation=0,  labelsize=7)
 
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved → {path}")
 
 
def plot_comparison(tokens, attn_scores, grad_scores, filename="comparison.png"):
    """Side-by-side comparison of attention vs gradient importance."""
    fig, axes = plt.subplots(2, 1, figsize=(max(10, len(tokens) * 0.6), 5))
    labels = ["Attention Rollout", "Gradient × Input"]
    scores = [attn_scores, grad_scores]
    cmaps  = [plt.cm.RdYlGn, plt.cm.YlOrRd]
 
    for ax, score, label, cmap in zip(axes, scores, labels, cmaps):
        colors = [cmap(s) for s in score]
        bars = ax.bar(range(len(tokens)), score, color=colors, edgecolor="white")
        ax.set_xticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Importance")
        ax.set_title(label, fontsize=11, fontweight="bold")
        ax.set_ylim(0, 1.1)
 
    plt.suptitle("Attention vs Gradient Importance Comparison", fontsize=13, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved → {path}")
 
 
def plot_bias_analysis(model, tokenizer, filename="bias_analysis.png"):
    """Check if the model assigns high importance to gendered or biased words."""
    biased_pairs = [
        ("The doctor examined his patient carefully.",
         "The nurse examined her patient carefully."),
        ("The engineer built the bridge successfully.",
         "The teacher helped the students learn effectively."),
        ("He was promoted for excellent leadership.",
         "She was promoted for excellent leadership."),
    ]
 
    fig, axes = plt.subplots(len(biased_pairs), 2,
                             figsize=(16, 3.5 * len(biased_pairs)))
 
    for row, (text_a, text_b) in enumerate(biased_pairs):
        for col, text in enumerate([text_a, text_b]):
            tokens, importance, pred = get_gradient_importance(model, tokenizer, text)
            ax = axes[row, col]
            ax.bar(range(len(tokens)), importance,
                   color=[plt.cm.YlOrRd(s) for s in importance])
            ax.set_xticks(range(len(tokens)))
            ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=7)
            ax.set_ylim(0, 1.1)
            label_str = "Positive" if pred == 1 else "Negative"
            ax.set_title(f'"{text[:45]}…"\nPrediction: {label_str}', fontsize=8)
 
    plt.suptitle("Bias Analysis: Gendered Pronoun Pairs", fontsize=13, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved → {path}")
 

## Main 

In [8]:
def main():
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
 
    # ── 1. Load Data ──────────────────────────────────────────────────────────
    train_ds, val_ds = load_data(tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
 
    # ── 2. Fine-tune BERT ─────────────────────────────────────────────────────
    print("\n── Fine-tuning BERT ──────────────────────────────────────────────")
    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, output_attentions=True
    ).to(DEVICE)
 
    num_params = sum(p.numel() for p in model.parameters())
    optimizer  = torch.optim.AdamW(model.parameters(), lr=LR)
 
    start = time.time()
    for epoch in range(1, EPOCHS + 1):
        loss = train(model, train_loader, optimizer)
        acc, f1 = evaluate(model, val_loader)
        print(f"Epoch {epoch}/{EPOCHS}  loss={loss:.4f}  acc={acc:.4f}  f1={f1:.4f}")
    training_time = time.time() - start
 
    # Final evaluation
    acc, f1 = evaluate(model, val_loader)
    print(f"\nFinal  →  Accuracy: {acc:.4f}  |  F1: {f1:.4f}")
    print(f"Training Time : {training_time:.1f}s")
    print(f"Parameters    : {num_params:,}")
 
    # ── 3. Example Interpretability ───────────────────────────────────────────
    sample_texts = [
        "This movie was absolutely wonderful and I loved every moment of it!",
        "The film was terrible and a complete waste of time.",
        "The acting was mediocre but the plot was surprisingly engaging.",
    ]
 
    for i, text in enumerate(sample_texts):
        print(f"\n── Sample {i+1}: {text[:60]}…")
 
        # ── Attention Rollout ──────────────────────────────────────────────
        tokens, attentions, seq_len = get_attention_weights(model, tokenizer, text)
        attn_scores = attention_rollout(attentions, seq_len)
 
        plot_token_heatmap(
            tokens, attn_scores,
            title=f'Attention Rollout — "{text[:50]}…"',
            filename=f"attn_heatmap_{i+1}.png",
            method="attention",
        )
        plot_attention_head_heatmap(
            attentions, tokens,
            layer=11, head=0,
            filename=f"attn_head_layer12_{i+1}.png",
        )
 
        # ── Gradient × Input ──────────────────────────────────────────────
        tokens_g, grad_scores, pred_class = get_gradient_importance(model, tokenizer, text)
        label_str = "Positive" if pred_class == 1 else "Negative"
        print(f"   Predicted: {label_str}")
 
        plot_token_heatmap(
            tokens_g, grad_scores,
            title=f'Gradient × Input — "{text[:50]}…" (Pred: {label_str})',
            filename=f"grad_heatmap_{i+1}.png",
            method="gradient",
        )
 
        # ── Comparison Plot ────────────────────────────────────────────────
        # Align tokens (attention and gradient may differ only in edge cases)
        min_len = min(len(tokens), len(tokens_g))
        plot_comparison(
            tokens[:min_len],
            attn_scores[:min_len],
            grad_scores[:min_len],
            filename=f"comparison_{i+1}.png",
        )
 
    # ── 4. Bias Analysis ──────────────────────────────────────────────────────
    print("\n── Running Bias Analysis …")
    plot_bias_analysis(model, tokenizer)
 
    # ── 5. Results Summary ────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print(" RESULTS SUMMARY")
    print("=" * 55)
    print(f" Model        : {MODEL_NAME} (fine-tuned)")
    print(f" Dataset      : SST-2 (GLUE)")
    print(f" Accuracy     : {acc:.4f}")
    print(f" F1 Score     : {f1:.4f}")
    print(f" Training Time: {training_time:.1f}s")
    print(f" Parameters   : {num_params:,}")
    print("=" * 55)
    print(f"\nAll visualizations saved to ./{OUTPUT_DIR}/")
 

In [9]:
if __name__ == "__main__":
    main()

Loading SST-2 dataset …


README.md: 0.00B [00:00, ?B/s]

C:\Users\sanka\.conda\envs\ai_master\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sanka\.cache\huggingface\hub\datasets--glue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

...

Epoch 1/2  loss=0.4832  acc=0.8875  f1=0.8875


Evaluating: 100%|██████████████████████████████████████████████████████████████████████| 25/25 [00:01<00:00, 20.31it/s]


Epoch 2/2  loss=0.1986  acc=0.8950  f1=0.8950


Evaluating: 100%|██████████████████████████████████████████████████████████████████████| 25/25 [00:01<00:00, 19.98it/s]



Final  →  Accuracy: 0.8950  |  F1: 0.8950
Training Time : 35.9s
Parameters    : 109,483,778

── Sample 1: This movie was absolutely wonderful and I loved every moment…
Saved → outputs\attn_heatmap_1.png
Saved → outputs\attn_head_layer12_1.png
   Predicted: Positive
Saved → outputs\grad_heatmap_1.png
Saved → outputs\comparison_1.png

── Sample 2: The film was terrible and a complete waste of time.…
Saved → outputs\attn_heatmap_2.png
Saved → outputs\attn_head_layer12_2.png
   Predicted: Negative
Saved → outputs\grad_heatmap_2.png
Saved → outputs\comparison_2.png

── Sample 3: The acting was mediocre but the plot was surprisingly engagi…
Saved → outputs\attn_heatmap_3.png
Saved → outputs\attn_head_layer12_3.png
   Predicted: Positive
Saved → outputs\grad_heatmap_3.png
Saved → outputs\comparison_3.png

── Running Bias Analysis …
Saved → outputs\bias_analysis.png

 RESULTS SUMMARY
 Model        : bert-base-uncased (fine-tuned)
 Dataset      : SST-2 (GLUE)
 Accuracy     : 0.8950
 F1 Score  